In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import numpy as np
import plotly.graph_objects as go

In [2]:
np.random.seed(23)

# Class 1
mu_vec1 = np.array([0, 0, 0])
cov_mat1 = np.eye(3)
class1_sample = np.random.multivariate_normal(mu_vec1, cov_mat1, 20)

df = pd.DataFrame(class1_sample, columns=['feature1', 'feature2', 'feature3'])
df['target'] = 1

mu_vec2 = np.array([1, 1, 1])
cov_mat2 = np.eye(3)
class2_sample = np.random.multivariate_normal(mu_vec2, cov_mat2, 20)

df1 = pd.DataFrame(class2_sample, columns=['feature1', 'feature2', 'feature3'])
df1['target'] = 0

df = pd.concat([df, df1], ignore_index=True)

# Shuffle rows
df = df.sample(frac=1, random_state=23).reset_index(drop=True)

df.head()

,feature1,feature2,feature3,target
0,-0.331617,-1.632386,0.619114,1
1,1.010229,1.437830,2.327788,0
2,0.241106,-0.952510,-0.136267,1
3,1.676860,4.187503,-0.080565,0
4,2.823378,-0.332863,2.637391,0


In [ ]:
fig = px.scatter_3d(df, x=df['feature1'], y=df['feature2'], z=df['feature3'],
              color=df['target'].astype('str'))

fig.update_traces(marker=dict(size=12,line=dict(width=2,color='DarkSlateGrey')),
                  selector=dict(mode='markers'))

fig.show()

### Step 1 - Apply standard scaling mean scaling

In [ ]:
scaler = StandardScaler()
df.iloc[:,0:3] = scaler.fit_transform(df.iloc[:,0:3])

### Step 2 - Find Covariance Matrix

In [9]:
covariance_matrix = np.cov([df.iloc[:,0],df.iloc[:,1],df.iloc[:,2]])
print('Covariance Matrix:\n', covariance_matrix)

Covariance Matrix:
 [[1.02564103 0.20478114 0.080118  ]
 [0.20478114 1.02564103 0.19838882]
 [0.080118   0.19838882 1.02564103]]


### Step 3 - Finding EV and EVs

In [10]:
eigen_values, eigen_vectors = np.linalg.eig(covariance_matrix)

In [ ]:
eigen_values

array([1.3536065 , 0.94557084, 0.77774573])

In [12]:
eigen_vectors

array([[-0.53875915, -0.69363291,  0.47813384],
       [-0.65608325, -0.01057596, -0.75461442],
       [-0.52848211,  0.72025103,  0.44938304]])

In [ ]:
# Extract mean
mean_point = df[['feature1', 'feature2', 'feature3']].mean().values

# Create scatter plot for points
scatter = go.Scatter3d(
    x=df['feature1'],
    y=df['feature2'],
    z=df['feature3'],
    mode='markers',
    marker=dict(size=4, opacity=0.6, color='blue'),
    name="Data Points"
)

# Mark the mean point
mean_scatter = go.Scatter3d(
    x=[mean_point[0]],
    y=[mean_point[1]],
    z=[mean_point[2]],
    mode='markers',
    marker=dict(size=6, color='red'),
    name="Mean"
)

# --- Eigenvector arrows ---
arrow_traces = []
for v in eigen_vectors.T:  # each eigenvector
    arrow = go.Scatter3d(
        x=[mean_point[0], mean_point[0] + v[0]],
        y=[mean_point[1], mean_point[1] + v[1]],
        z=[mean_point[2], mean_point[2] + v[2]],
        mode='lines',
        line=dict(width=6, color='red'),
        name="Eigenvector"
    )
    arrow_traces.append(arrow)

# Create the figure
fig = go.Figure(data=[scatter, mean_scatter] + arrow_traces)

fig.update_layout(
    title="3D Eigenvectors Visualization (Plotly)",
    scene=dict(
        xaxis_title="Feature 1",
        yaxis_title="Feature 2",
        zaxis_title="Feature 3"
    ),
    width=800,
    height=700
)

fig.show()

### We take the top 2 vectors PC1,2 these 2 vectors will be used for the linear transformation. 

In [ ]:
pc = eigen_vectors[0:2]
pc

array([[-0.53875915, -0.69363291,  0.47813384],
       [-0.65608325, -0.01057596, -0.75461442]])

In [17]:
transformed_df = np.dot(df.iloc[:,0:3],pc.T)
# 40,3 - 3,2
new_df = pd.DataFrame(transformed_df,columns=['PC1','PC2'])
new_df['target'] = df['target'].values
new_df.head()

,PC1,PC2,target
0,1.726114,0.492511,1
1,-0.220797,-1.441911,0
2,0.688605,0.658084,1
3,-3.367715,-0.254627,0
4,0.227326,-2.669841,0


#### Plotting PC1,2

In [18]:
new_df['target'] = new_df['target'].astype('str')
fig = px.scatter(x=new_df['PC1'],
                 y=new_df['PC2'],
                 color=new_df['target'],
                 color_discrete_sequence=px.colors.qualitative.G10
                )

fig.update_traces(marker=dict(size=12,
                              line=dict(width=2,
                                        color='DarkSlateGrey')),
                  selector=dict(mode='markers'))
fig.show()